# 00 · 总览：为什么重做，怎么重做

这本笔记一步一章。每章固定五段：**做了什么 → 工业界叫什么 → 为什么 → ✏️ 你来验证 → 思考**。
灰色的代码格可以直接点运行（左上角 ▶ 或 Shift+Enter），输出就在格子下面。

这一章没有命令，只有地图。

## 这个项目要做什么

给一个病人的症状（可选：测序结果），agent 自己决定查什么，排出最可能的罕见病和致病基因，
每条结论附证据；然后在严格防泄漏的公开病例上，和传统方法、固定流程、DeepRare 比较。
完整计划见 `docs/00_plan.md`。

它接着 OtoVCE 那篇论文往下做。那篇的两个问题——不是真正的 agent、只做耳聋——是这个项目的起点。

## 旧仓库的问题（面试官五分钟内会看到的）

| 现象 | 数字 |
|---|---|
| 提交太少 | 5 个月 9 次提交；最后一次之后两个月的工作没进版本库 |
| 几代代码叠着 | 约 390 个 Python 文件、近 8 万行，step1–5、src、otovce、otovce_v2、curator 五套并存 |
| 同一段逻辑抄了多遍 | 调工具的主循环在 8 个文件里各写一遍 |
| 脚本当程序用 | 64 个脚本；88 个文件靠手动改搜索路径才能互相找到 |
| 数据混在代码里 | 22 个基因组索引文件在根目录并进了 git |
| 检查只覆盖一角 | 自动检查只管 otovce_v2 一个目录 |
| 没人真跑过 | Dockerfile 启动要 uvicorn，依赖清单里没有 |

底子不差：防泄漏、预注册、对照组这套评测纪律是真的。缺的是收纳，不是能力。

## 为什么新开仓库而不是原地改

1. 旧仓库是已投稿论文的公开代码，审稿期间要能原样复现。
2. 题目换了（耳聋变异分类 → 罕见病诊断），旧代码大部分用不上。
3. 提交历史本身是作品。新仓库从第一天起每一步都是"小改动 + 自动检查通过"，旧仓库补不回来。

## 路线：和公司里的顺序一样

先有尺子，再做工具，再做 agent，再上线。

| 步 | 做什么 | 工业界叫什么 | 做完能看到 |
|---|---|---|---|
| 0 | 旧仓库打标签存档 | release tag | 论文代码有固定版本 |
| 1 | 新仓库骨架：标准目录、依赖锁定、自动检查、Docker | walking skeleton | GitHub 上第一个绿勾 |
| 2 | 评测底座：读病例、防泄漏、判分、一条命令出报告；跑"大模型直接猜" | eval harness | 第一个准确率数字 |
| 3 | 工具 T1–T4，每个有规格和测试，包成 MCP 服务 | tools, MCP | 在 Claude Code 里直接调用我们的工具 |
| 4 | agent：LangGraph 循环、结束前复核、全程追踪 | orchestration, tracing | 逐步看它调了什么；agent 对固定流程的对比表 |
| 5 | 做成服务：接口、网页、一条命令启动；评测进 CI 当闸门 | serving, regression gate | 换台电脑一条命令跑起来 |
| 6 | 放大：基因型工具、万例评测、DeepRare 对照 | large-scale eval | 论文主表 |
| 7 | 交付：论文、README、架构图、演示视频 | release | 可投稿、可投简历 |

做完第 5 步就能拿去面试，不用等万例评测跑完。

## "agent 有没有 Databricks 那样的平台"

有，而且和 Databricks 一样：**代码还是人写，平台管写完之后的事**（跑、记录、评测、对比版本）。
你在 qafi-data-platform 里写的 106 个 dbt SQL 文件就是代码，Databricks 只是跑它们的地方。

| Databricks 给数据工程的 | agent 这边 | 我们用 |
|---|---|---|
| 写代码：notebook、dbt | 代码编辑器 | VS Code |
| 存代码：Git folders 连 GitHub | GitHub | GitHub |
| 跑：Jobs | 本机、Docker、云 | 本机 + Docker |
| 看运行、做评测：日志、MLflow | LangSmith / Langfuse | 第 4 步二选一 |

真正不写代码的是 Dify、Coze 那类拖拽平台，像试剂盒：快，配方改不了。工程岗考的是自己开发方法。

## 公司怎么管代码才不会写乱

不靠"小心别写乱"，靠流程让乱不起来，乱了也能退回去。

| 规矩 | 大白话 | 像实验室里的 |
|---|---|---|
| 版本控制（Git） | 每次改动记下谁、何时、为什么；随时能退回任一版 | 实验记录本，不能撕页 |
| 分支 + 合并请求（branch / PR） | 不直接改正本；在副本上改完提申请并入 | 草稿上改方案，过目后誊进正本 |
| 自动检查（CI） | 每个申请自动跑格式、类型、测试，红的进不去 | 每批样本必须过的 QC |
| 代码审查（code review） | 第二个人看过才能并入 | 记录要有人复核签字 |
| 固定结构 | 逻辑都在一个包里；脚本只能是一行调用 | 试剂有固定柜位，不堆台面 |
| 小步改动 | 一次只改一件事，几十到几百行 | 每次只改方案的一处 |

另外两条：数据和运行结果不进代码库（样本放冰箱，不夹在记录本里）；没用的代码直接删，历史都在 Git 里。

## 不用看代码也能判断乱没乱

1. 每个合并请求有没有绿勾。
2. 每个合并请求改了多少行，超过几百行就该拆。
3. 目录里有没有长出一堆新脚本。

想看真实的样子：打开 github.com/langchain-ai/langgraph，点 Pull requests，每一条就是这套流程在跑。